# GTEx model building with GenomicSuperSignature

💡 **Environment:** `clamp-analyses`  

## Libraries

In [4]:
library(here)
library(matrixStats)
library(factoextra)
library(cluster)
library(GenomicSuperSignature)
library(msigdbr)
library(fgsea)
library(dplyr)
library(tibble)

## Input

In [5]:
gtex_data <- readRDS(here('output/gtex/df_gtex_fbm_filt.rds'))
head(gtex_data)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,1.44614308,0.1466590,1.15444418,1.8411744,-0.07922225,0.6559264,1.9780371,2.5806060,1.6749832,2.04978755,⋯,-1.0148940,-0.59314323,0.65505136,-0.5693505,-0.48892918,0.2077396965,-0.8016505,-0.28439422,-1.6336384,-0.65163455
RP11-34P13.15,-0.21703290,-1.2491072,-0.71067536,-0.5458638,-0.88156172,-0.4026354,-0.5965898,0.2922380,-0.4949213,0.05590706,⋯,-0.1934056,0.33783682,1.59084836,0.4045636,0.05473854,1.0872298726,0.5077824,-0.19027836,-0.9777920,0.24859783
RP11-34P13.16,0.05000362,-1.2529810,-1.26148519,-0.7514709,-0.77242737,-0.3766090,-0.9801061,0.4096420,-0.3255995,-0.06358077,⋯,0.1692925,0.70250273,1.43810645,0.6769784,0.17310765,1.2674193527,0.8571877,-0.10447332,-1.0448862,0.46041845
RP11-34P13.18,0.80964288,-0.4648762,0.60984294,0.6852051,-1.14329360,-0.5356296,1.4716736,1.0668942,0.1565502,0.90028575,⋯,-1.3100707,-0.67305864,-0.05994370,-1.0821739,-0.70715756,0.0894613755,-1.3990557,-0.65677160,-1.7141016,-0.37942569
AP006222.2,0.60340589,-0.6590192,-0.22788724,-0.3186479,-0.75919490,-0.8022825,-0.8073561,-0.9128508,-0.3943123,-0.90819239,⋯,0.1568485,1.49184552,1.31865074,-0.1807585,0.13219755,2.1816052692,-0.3754781,-0.03196266,-0.2097524,3.06398139
MTND1P23,-0.28701222,0.4584699,-0.04655404,-0.6834497,-0.28493094,0.5112355,-0.4015529,-0.5931863,0.1299671,-0.25562386,⋯,1.0400459,-0.03813723,-0.04317319,-0.1047992,0.06352471,0.0003105984,-0.2519768,-0.01557983,0.3048702,-0.02531932


## PCA

In [ ]:
n <- readRDS(here("output/gtex/CLAMP_K_gtex.rds"))
study <- 'GTEx'
d <- 4

In [7]:
pca_res <- prcomp(t(as.matrix(gtex_data)))   # x is a matrix with genes(row) x samples(column)

In [8]:
trainingData_PCA <- list()
trainingData_PCA[[study]] <- list()

trainingData_PCA[[study]]$rotation <- pca_res$rotation[, 1:n]
colnames(trainingData_PCA[[study]]$rotation) <- paste0(study, ".PC", 1:n)

In [9]:
eigs <- pca_res$sdev^2

In [10]:
pca_summary <- rbind(SD = sqrt(eigs),
                   Variance = eigs/sum(eigs),
                   Cumulative = cumsum(eigs)/sum(eigs))

In [11]:
trainingData_PCA[[study]]$variance <- pca_summary[,1:n]
colnames(trainingData_PCA[[study]]$variance) <- paste0(study, ".PC", c(1:n))

## Hierarchical clustering

In [12]:
allZ <- trainingData_PCA[[study]]$rotation
storage.mode(allZ) <- "double"
all  <- t(allZ)

In [13]:
res.dist <- factoextra::get_dist(all, method = "spearman")

In [14]:
# Cut the tree
k <- round(nrow(all)/d, 0)
res.hcut <- factoextra::hcut(res.dist, k = k, hc_func = "hclust", 
                             hc_method = "ward.D", hc_metric = "spearman")

In [15]:
# Build avgLoading 
trainingData_PCclusters <- buildAvgLoading(allZ, k, cluster = res.hcut$cluster)

In [16]:
# Silhouette Width
cl <- trainingData_PCclusters$cluster
silh_res <- cluster::silhouette(cl, res.dist)
cl_silh_width <- summary(silh_res)$clus.avg.widths
trainingData_PCclusters$sw <- cl_silh_width  # add silhouette width to the result

## Final model

In [17]:
trainingData_df <- DataFrame(
  PCAsummary = I(list(trainingData_PCA[[study]]$variance))
)
rownames(trainingData_df) <- study

In [18]:
# Construct PCAGenomicSignatures
RAVmodel <- PCAGenomicSignatures(
  assays       = list(RAVindex = as.matrix(trainingData_PCclusters$avgLoading)),
  trainingData = trainingData_df
)

In [19]:
# Attach metadata analogous to the multi-study build
metadata(RAVmodel) <- trainingData_PCclusters[c("cluster","size","k","n")]
names(metadata(RAVmodel)$size) <- paste0("RAV", seq_len(ncol(RAVmodel)))

geneSets(RAVmodel)        <- "Custom"                          # label as you wish
studies(RAVmodel)         <- trainingData_PCclusters$studies   # PC->study map
silhouetteWidth(RAVmodel) <- trainingData_PCclusters$sw
updateNote(RAVmodel)      <- paste0("Single-matrix GTEx model; PCs = ", n, ".")
metadata(RAVmodel)$version <- "0.1.0-single"

RAVmodel

class: PCAGenomicSignatures 
dim: 21613 103 
metadata(7): cluster size ... updateNote version
assays(1): RAVindex
rownames(21613): WASH7P RP11-34P13.15 ... MT-TT MT-TP
rowData names(0):
colnames(103): Cl103_01 (3/1) Cl103_02 (4/1) ... Cl103_102 (2/1)
  Cl103_103 (2/1)
colData names(2): studies silhouetteWidth
trainingData(1): PCAsummary
trainingData names(1): GTEx

In [20]:
msig_category <- "C2" 
msig_df <- msigdbr(species = "Homo sapiens", category = msig_category)
pathways <- split(msig_df$gene_symbol, msig_df$gs_name)

RAVindex <- as.matrix(trainingData_PCclusters$avgLoading)
stopifnot(!is.null(rownames(RAVindex)))
rav_names <- colnames(RAVindex)

prep_ranks <- function(v) {
  v <- v[is.finite(v)]                 
  v <- tapply(v, names(v), function(x) x[which.max(abs(x))]) |> unlist()
  sort(v, decreasing = TRUE)
}

gsea_list <- vector("list", length(rav_names))
names(gsea_list) <- rav_names

for (j in seq_along(rav_names)) {
  ranks <- RAVindex[, j]
  names(ranks) <- rownames(RAVindex)
  ranks <- prep_ranks(ranks)
  stype <- if (all(ranks >= 0)) "pos" else if (all(ranks <= 0)) "neg" else "std"

  res <- fgsea(
    pathways = pathways,
    stats    = ranks,
    minSize  = 10,
    maxSize  = 5000,
    scoreType = stype           
  ) |>
    arrange(padj, desc(NES)) |>
    as_tibble()

  gsea_list[[j]] <- res
}

gsea(RAVmodel) <- gsea_list
metadata(RAVmodel)$gsea_collection <- msig_category
metadata(RAVmodel)$version <- paste0(metadata(RAVmodel)$version, "+gsea")

Warning message:
“The `category` argument of `msigdbr()` is deprecated as of msigdbr 10.0.0.
ℹ Please use the `collection` argument instead.”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 69 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval, padj, NES, log2err are set to NA. You can try to increase the value of the argument nPermSimple (for example set it nPermSimple = 10000)”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some of the pathways the P-values were likely overestimated. For such pathways log2err is set to NA.”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some pathways, in reality P-values are less than 1e-50. You can set the `eps` argument to zero for better estimation.”
Warning message in fgseaMultilevel(pathways = 

In [21]:
B <- assays(RAVmodel)[["RAVindex"]]
B_df <- as.data.frame(B)
head(B_df)

,Cl103_01 (3/1),Cl103_02 (4/1),Cl103_03 (8/1),Cl103_04 (6/1),Cl103_05 (3/1),Cl103_06 (6/1),Cl103_07 (2/1),Cl103_08 (4/1),Cl103_09 (4/1),Cl103_10 (2/1),⋯,Cl103_94 (4/1),Cl103_95 (4/1),Cl103_96 (3/1),Cl103_97 (4/1),Cl103_98 (4/1),Cl103_99 (2/1),Cl103_100 (2/1),Cl103_101 (4/1),Cl103_102 (2/1),Cl103_103 (2/1)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,0.004189115,0.002797852,-0.0006309973,-0.0034582017,0.0013606441,0.001843070,0.0046842141,1.373732e-03,0.0019847772,-0.003480212,⋯,-0.0042188056,-0.0039268555,-0.001587174,0.001773409,-0.0050495481,0.002700624,0.0069806229,-0.003222641,0.0099607598,0.0151296520
RP11-34P13.15,-0.004590269,0.005326955,0.0013098786,0.0016191595,-0.0040962991,-0.001746955,-0.0012789397,1.229684e-03,0.0008024653,-0.002132374,⋯,0.0025640274,-0.0033307169,-0.009569394,-0.001872299,-0.0006982200,-0.001362159,0.0021862807,-0.008333364,0.0053719645,-0.0022115768
RP11-34P13.16,-0.006140078,0.005775901,0.0025717016,0.0012099495,-0.0050306843,-0.003245759,-0.0042933791,-4.119338e-05,-0.0006394154,0.002469818,⋯,0.0021210501,-0.0032871824,-0.011146120,-0.001956016,0.0002121118,-0.003906272,0.0020183460,-0.008652569,0.0061547785,-0.0003294824
RP11-34P13.18,0.003143366,0.002920637,0.0045881832,-0.0037703402,0.0006174415,0.004354348,0.0056581128,1.188683e-03,-0.0003049506,-0.001354101,⋯,0.0007416665,0.0013269786,-0.007158309,-0.002701745,-0.0013669957,-0.001763925,0.0002691804,0.003865256,0.0001805685,-0.0033327603
AP006222.2,-0.003184713,0.004652395,0.0037706099,-0.0002310365,-0.0042249683,0.004525349,0.0048568631,2.659670e-03,-0.0021605683,-0.009422430,⋯,0.0092542088,0.0047298075,-0.010044632,-0.034946478,-0.0002753004,-0.006338076,0.0153168130,-0.007576892,-0.0046235272,-0.0302953426
MTND1P23,-0.003064182,-0.003572230,-0.0031347251,0.0003719925,0.0024576307,-0.000277947,0.0009606978,3.289754e-04,0.0007254853,0.004074466,⋯,-0.0130899611,-0.0009645902,0.005354001,-0.013667997,0.0063602282,-0.018094015,-0.0016912048,0.010655997,0.0178552731,0.0045448310


In [22]:
library(GenomicSuperSignature)

# genes × samples (numeric matrix)
expr <- as.matrix(gtex_data)
storage.mode(expr) <- "double"
rownames(expr) <- make.unique(rownames(expr))

# loadings from model: genes × RAVs
RAVindex <- assays(RAVmodel)[["RAVindex"]] |> as.matrix()
storage.mode(RAVindex) <- "double"

# align by genes (same order in both)
common <- sort(intersect(rownames(expr), rownames(RAVindex)))
expr_c     <- expr[common, , drop = FALSE]
RAVindex_c <- RAVindex[common, , drop = FALSE]

# RAV × sample scores (B in LV-space)
B_RAVxSample <- crossprod(RAVindex_c, expr_c)   # == t(RAVindex_c) %*% expr_c

# OPTIONAL: reconstruct gene × sample from model
X_hat <- RAVindex_c %*% B_RAVxSample

# quick checks
dim(expr_c)        # genes × samples (input)
dim(RAVindex_c)    # genes × RAVs
dim(B_RAVxSample)  # RAVs × samples
dim(X_hat)         # genes × samples (reconstruction)

[1] 21613 17382

[1] 21613   103

[1]   103 17382

[1] 21613 17382

In [23]:
head(B_RAVxSample)
dim(B_RAVxSample)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
Cl103_01 (3/1),20.1436331,-27.119669,16.032595,28.7370463,-26.5056742,-7.807362,38.1813619,19.463378,10.306126,12.0203243,⋯,-19.7656241,6.9970846,25.104542,-6.3537829,3.315612,34.250080,-7.215533,15.114328,-26.7105151,9.264905
Cl103_02 (4/1),12.2482518,-8.000599,2.648479,-0.8847262,-2.4760072,10.064249,0.9034389,19.584220,8.847291,13.1333473,⋯,-10.7340885,-6.6352337,6.221537,6.1647014,-11.394615,-6.014258,18.265955,-13.955039,-17.1869842,4.732763
Cl103_03 (8/1),2.3490163,-6.590285,2.366034,4.9573081,2.4658449,3.092498,4.8453042,-10.434050,2.093771,-4.2724657,⋯,0.7021393,-0.7515066,3.397561,0.1024156,2.341242,1.714378,-9.155160,3.590645,-3.4930188,2.935004
Cl103_04 (6/1),0.4122997,-9.332884,-4.723007,-1.1511334,-6.9020091,1.235822,-12.8657181,-5.343738,-1.434950,0.5243584,⋯,3.2801029,4.0533179,-14.284605,-4.7776558,3.803543,-14.309870,4.463534,7.388846,-0.2500049,9.842788
Cl103_05 (3/1),2.5543271,-7.114111,-1.656403,-1.6203822,-0.1347983,6.360862,-6.4418527,-5.017466,4.579001,0.7933866,⋯,-2.5149476,-3.6312141,-3.310292,7.1765218,11.219318,-19.242411,-9.130031,0.282605,-0.8521094,9.623557
Cl103_06 (6/1),-5.6137659,-0.601384,-3.609692,2.6047424,-1.5239880,1.785555,5.3278155,10.235236,3.086108,4.4576902,⋯,-5.3018638,-3.5782848,-1.955674,2.6429339,3.211387,8.099340,6.670341,4.383245,2.3252346,4.723493


[1]   103 17382

In [24]:
output_dir <- here("output/gtex/GenomicSuperSignature")
dir.create(output_dir, showWarnings = FALSE)

In [25]:
write.csv(B_RAVxSample,
          file = here(output_dir, "gtex_B.csv"),
          quote = FALSE)